#  Marketing Qualified Leads - Bronze Ingestion


## Imports

In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, DateType, StructField, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "olist_marketing_qualified_leads"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "olist_marketing"
source_dataset = "marketing_qualified_leads"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("mql_id", StringType(), True),
    StructField("first_contact_date", DateType(), True),
    StructField("landing_page_id", StringType(), True),
    StructField("origin", StringType(), True)
])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- mql_id: string (nullable = true)
 |-- first_contact_date: date (nullable = true)
 |-- landing_page_id: string (nullable = true)
 |-- origin: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

mql_id,first_contact_date,landing_page_id,origin,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
dac32acd4db4c29c230538b72f8dd87d,2018-02-01,88740e65d5d6b056e0cda098e1ea6313,social,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
8c18d1de7f67e60dbd64e3c07d7e9d5d,2017-10-20,007f9098284a86ee80ddeb25d53e0af8,paid_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
b4bc852d233dfefc5131f593b538befa,2018-03-22,a7982125ff7aa3b2054c6e44f9d28522,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
6be030b81c75970747525b843c1ef4f8,2018-01-22,d45d558f0daeecf3cccdffe3c59684aa,email,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads
5420aad7fec3549a85876ba1c529bd84,2018-02-21,b48ec5f3b04e9068441002a19df93c6c,organic_search,null,/Volumes/ecommerce_dev/landing/raw_files/olist_marketing/marketing_qualified_leads/olist_marketing_qualified_leads_dataset.csv,2026-08-02T21:45:36.000Z,2026-08-03T01:40:01.839Z,4055dcbe-6064-419a-9c4e-27cd76475192,olist_marketing,marketing_qualified_leads


In [0]:
spark.table(target_table).count()

8000